In [1]:
import pandas as pd
import numpy as np
import glob
import os
from dotenv import load_dotenv
import glob

In [2]:
path_input = os.getenv('path_input')

path_output = os.getenv('path_output')

In [ ]:
# Spotify's timestamp is in UTC. Change the utc_offset variable to whatever the UTC offset is for your location.

utc_offset = -4

In [4]:
def convert_milliseconds(row):
    total_seconds, remaining_ms = divmod(row["ms_played"], 1000)
    minutes, seconds = divmod(total_seconds, 60)
    row["TimePlayed"] = f"{minutes}:{seconds:02}"
    return row["TimePlayed"]   

In [5]:
def import_spotify_data(df):
    spotify_json_files = glob.glob(path_input + "*.json")
    df_list = (pd.read_json(file, dtype="string") for file in spotify_json_files)
    df = pd.concat(df_list, ignore_index=True)
    return df

In [6]:
def process_spotify_data(df):
    df["ts"] = pd.to_datetime(df['ts'])
    df["ts"] = df["ts"] + pd.Timedelta(hours=utc_offset)
    df["Date"] = pd.to_datetime(df["ts"]).dt.date
    df["Time"] = pd.to_datetime(df["ts"]).dt.strftime('%-I:%M %p')
    df["ms_played"] = df["ms_played"].astype(int)
    df["TimePlayed"] = df.apply(convert_milliseconds, axis=1)
    df = df[["Date", "Time", "master_metadata_track_name", "master_metadata_album_artist_name", "master_metadata_album_album_name", "TimePlayed"]]
    df = df.rename(columns={'master_metadata_track_name': 'Track', 'master_metadata_album_artist_name': 'Artist', 'master_metadata_album_album_name': 'Album', "TimePlayed":"Duration"})
    return df


In [7]:
def export_data(df):
    all_dates = df['Date'].unique().tolist()
    for day in all_dates:
        df_date = df[(df['Date'] == day)]
        day = day.strftime('%Y-%m-%d')
        df_date.to_csv(path_output + "Spotify-" + day + ".csv", index=False)
        markdown_str = df_date.to_markdown(index=False)
        with open(path_output + "Spotify-" + day + ".md", "w") as file:
            file.write(markdown_str)

In [8]:
spotify_data = pd.DataFrame

spotify_data = import_spotify_data(spotify_data)

In [9]:
spotify_data = process_spotify_data(spotify_data)

In [10]:
export_data(spotify_data)